# Image classification with Transformers

Follow the Hugging Face Transformers image classification tutorial while working through this notebook:

https://huggingface.co/docs/transformers/en/tasks/image_classification

The code below follows that tutorial closely. The main change is the dataset: instead of Food-101, this notebook uses [AQUA20](https://huggingface.co/datasets/taufiktrf/AQUA20), an underwater image classification dataset with 20 labels.


## Setup

Install the dependencies from `requirements.txt` before running the notebook.


In [ ]:
!pip install -q -r https://raw.githubusercontent.com/stefanluedtke/2026-RoOT/main/requirements.txt



## Load AQUA20 dataset

Start by loading a smaller subset of AQUA20. This is useful for checking that the full training pipeline works before training on more images.


In [ ]:
from datasets import load_dataset

# Use a subset first. For a full run, load the full train split instead.
aqua = load_dataset("taufiktrf/AQUA20", split="train[:5000]")


Split the dataset into train and test sets with `train_test_split`.


In [ ]:
aqua = aqua.train_test_split(test_size=0.2, seed=42)
aqua


Take a look at one example. Each example has an `image` and a `label`.


In [ ]:
aqua["train"][0]


Create dictionaries that map between label ids and label names.


In [ ]:
labels = aqua["train"].features["label"].names
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

id2label[str(0)], len(labels)


## Task 1

Use the Hugging Face tutorial section **Load Food-101 dataset** as a guide and answer these questions:

1. How many training examples and test examples are in `aqua`?
2. What is the label name of the first training example?
3. Display the first training image.


In [ ]:
# Your code here


## Preprocess

Load an image processor and define the image transforms. This follows the preprocessing section of the Hugging Face tutorial.


In [ ]:
from transformers import AutoImageProcessor

checkpoint = "google/vit-base-patch16-224-in21k"
image_processor = AutoImageProcessor.from_pretrained(checkpoint)


In [ ]:
from torchvision.transforms import RandomResizedCrop, Compose, Normalize, ToTensor

normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
size = (
    image_processor.size["shortest_edge"]
    if "shortest_edge" in image_processor.size
    else (image_processor.size["height"], image_processor.size["width"])
)
_transforms = Compose([RandomResizedCrop(size), ToTensor(), normalize])


In [ ]:
def transforms(examples):
    examples["pixel_values"] = [_transforms(img.convert("RGB")) for img in examples["image"]]
    del examples["image"]
    return examples


aqua = aqua.with_transform(transforms)


In [ ]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()


## Task 2

Use the Hugging Face tutorial section **Preprocess** as a guide.

1. Inspect one transformed example from `aqua["train"]`.
2. Check the shape of its `pixel_values` tensor.
3. Explain why `remove_unused_columns=False` will be needed later when training with `Trainer`.


In [ ]:
# Your code here


## Evaluate

Load the `accuracy` metric and create a `compute_metrics` function.


In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)


## Train

Load the model and set up `TrainingArguments` and `Trainer`.


In [ ]:
from transformers import AutoModelForImageClassification, TrainingArguments, Trainer

model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)


In [ ]:
from inspect import signature

# Transformers renamed evaluation_strategy to eval_strategy in newer versions.
eval_argument_name = "eval_strategy" if "eval_strategy" in signature(TrainingArguments.__init__).parameters else "evaluation_strategy"

training_args = TrainingArguments(
    output_dir="aqua20-vit",
    remove_unused_columns=False,
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    warmup_ratio=0.1,
    logging_steps=10,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    **{eval_argument_name: "epoch"},
)


In [ ]:
trainer_kwargs = dict(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=aqua["train"],
    eval_dataset=aqua["test"],
    compute_metrics=compute_metrics,
)

# Current Transformers uses processing_class. Older versions used tokenizer or no processor argument.
if "processing_class" in signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = image_processor

trainer = Trainer(**trainer_kwargs)


## Task 3

Use the Hugging Face tutorial section **Train** as a guide.

1. Fine-tune the model with `trainer.train()`.
2. Evaluate it with `trainer.evaluate()`.
3. Increase the number of training examples or epochs and compare the accuracy.


In [ ]:
# Your code here


In [ ]:
# Your code here


In [ ]:
# Your code here


## Inference

Use the fine-tuned model for inference. This follows the inference section of the Hugging Face tutorial.


In [ ]:
from transformers import pipeline

# Use the local output directory after training has completed.
classifier = pipeline("image-classification", model="aqua20-vit", image_processor=image_processor)

raw_example = load_dataset("taufiktrf/AQUA20", split="test[:1]")[0]
image = raw_example["image"]
classifier(image)


In [ ]:
import torch
from transformers import AutoModelForImageClassification

image_processor = AutoImageProcessor.from_pretrained("aqua20-vit")
inputs = image_processor(image, return_tensors="pt")

model = AutoModelForImageClassification.from_pretrained("aqua20-vit")
with torch.no_grad():
    logits = model(**inputs).logits

predicted_label = logits.argmax(-1).item()
model.config.id2label[predicted_label]


## References

- Hugging Face image classification tutorial: https://huggingface.co/docs/transformers/en/tasks/image_classification
- AQUA20 dataset: https://huggingface.co/datasets/taufiktrf/AQUA20
